In [ ]:
# Nallan Chakravathula Yashaswini
# Contact Management System
# Week 3 Project - Functions & Dictionaries

import json
import re
from datetime import datetime
import csv

DATA_FILE = "contacts_data.json"

def validate_phone(phone):
    """Validate phone number format"""
    digits = re.sub(r'\D', '', phone)
    if 10 <= len(digits) <= 15:
        return True, digits
    return False, None

def validate_email(email):
    """Validate email format"""
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def add_contact(contacts):
    """Add a new contact to the dictionary"""
    print("\n--- ADD NEW CONTACT ---")
    
    # Get contact name
    while True:
        name = input("Enter contact name: ").strip()
        if name:
            if name in contacts:
                print(f"Contact '{name}' already exists!")
                choice = input("Do you want to update instead? (y/n): ").lower()
                if choice == 'y':
                    update_contact(contacts, name)
                    return contacts
            break
        print("Name cannot be empty!")
    
    # Get phone number with validation
    while True:
        phone = input("Enter phone number: ").strip()
        is_valid, cleaned_phone = validate_phone(phone)
        if is_valid:
            break
        print("Invalid phone number! Please enter 10-15 digits.")
    
    # Get email with validation
    while True:
        email = input("Enter email (optional, press Enter to skip): ").strip()
        if not email or validate_email(email):
            break
        print("Invalid email format!")
    
    # Get additional info
    address = input("Enter address (optional): ").strip()
    group = input("Enter group (Friends/Work/Family/Other): ").strip() or "Other"
    
    # Store in dictionary
    now = datetime.now().isoformat()
    contacts[name] = {
        'phone': cleaned_phone,
        'email': email if email else None,
        'address': address if address else None,
        'group': group,
        'created_at': now,
        'updated_at': now
    }
    
    print(f"✅ Contact '{name}' added successfully!")
    return contacts

def search_contacts(contacts, search_term):
    """Search contacts by name (partial match)"""
    search_term = search_term.lower()
    results = {}
    
    for name, info in contacts.items():
        if search_term in name.lower():
            results[name] = info
    
    return results

def display_search_results(results):
    """Display search results in formatted way"""
    if not results:
        print("No contacts found.")
        return
    
    print(f"\nFound {len(results)} contact(s):")
    print("-" * 50)
    
    for i, (name, info) in enumerate(results.items(), 1):
        phone = info.get('phone', '-')
        email = info.get('email')
        address = info.get('address')
        group = info.get('group', 'Other')
        
        print(f"{i}. {name}")
        print(f"   📞 Phone: {phone}")
        if email:
            print(f"   📧 Email: {email}")
        if address:
            print(f"   📍 Address: {address}")
        print(f"   👥 Group: {group}")
        print()
        
def display_all_contacts(contacts):
    """Display all contacts"""
    if not contacts:
        print("No contacts found.")
        return
    print("\n--- ALL CONTACTS ---")
    display_search_results(contacts)

def update_contact(contacts, name=None):
    """Update an existing contact"""
    print("\n--- UPDATE CONTACT ---")
    
    if not contacts:
        print("No contacts to update.")
        return contacts
    
    if not name:
        search_term = input("Enter name to search and update: ").strip()
        results = search_contacts(contacts, search_term)
        display_search_results(results)
        if not results:
            return contacts
        name = input("Enter exact name to update: ").strip()
    
    if name not in contacts:
        print(f"Contact '{name}' not found.")
        return contacts
    
    info = contacts[name]
    
    print(f"Current phone: {info['phone']}")
    new_phone = input("New phone (press Enter to keep): ").strip()
    if new_phone:
        is_valid, cleaned = validate_phone(new_phone)
        if is_valid:
            info['phone'] = cleaned
        else:
            print("Invalid phone. Keeping old phone.")
    
    print(f"Current email: {info['email']}")
    new_email = input("New email (press Enter to keep): ").strip()
    if new_email:
        if validate_email(new_email):
            info['email'] = new_email
        else:
            print("Invalid email. Keeping old email.")
    
    print(f"Current address: {info['address']}")
    new_address = input("New address (press Enter to keep): ").strip()
    if new_address:
        info['address'] = new_address
    
    print(f"Current group: {info['group']}")
    new_group = input("New group (press Enter to keep): ").strip()
    if new_group:
        info['group'] = new_group
    
    info['updated_at'] = datetime.now().isoformat()
    contacts[name] = info
    print(f"✅ Contact '{name}' updated successfully!")
    return contacts

def delete_contact(contacts):
    """Delete a contact"""
    print("\n--- DELETE CONTACT ---")
    
    if not contacts:
        print("No contacts to delete.")
        return contacts
    
    search_term = input("Enter name to search and delete: ").strip()
    results = search_contacts(contacts, search_term)
    display_search_results(results)
    if not results:
        return contacts
    
    name = input("Enter exact name to delete: ").strip()
    if name not in contacts:
        print(f"Contact '{name}' not found.")
        return contacts
    
    confirm = input(f"Are you sure you want to delete '{name}'? (y/n): ").lower().strip()
    if confirm == 'y':
        del contacts[name]
        print(f"🗑️ Contact '{name}' deleted successfully!")
    else:
        print("Deletion cancelled.")
    
    return contacts

def load_contacts():
    """Load contacts from JSON file"""
    try:
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}
        
   # Ensure all keys exist
    for info in contacts.values():
        info.setdefault('phone', '')
        info.setdefault('email', None)
        info.setdefault('address', None)
        info.setdefault('group', 'Other')
        info.setdefault('created_at', datetime.now().isoformat())
        info.setdefault('updated_at', datetime.now().isoformat())
    return contacts

def save_contacts(contacts):
    """Save contacts to JSON file"""
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(contacts, f, indent=4)

def export_to_csv(contacts, filename="contacts_export.csv"):
    """Export contacts to CSV"""
    if not contacts:
        print("No contacts to export.")
        return
    
    fieldnames = ["name", "phone", "email", "address", "group", "created_at", "updated_at"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for name, info in contacts.items():
            row = {"name": name}
            row.update(info)
            writer.writerow(row)
    
    print(f"📁 Contacts exported to {filename}")

def show_statistics(contacts):
    """Show simple statistics"""
    total = len(contacts)
    groups = {}
    for info in contacts.values():
        g = info.get("group", "Other")
        groups[g] = groups.get(g, 0) + 1
    
    print("\n--- STATISTICS ---")
    print(f"Total contacts: {total}")
    for g, count in groups.items():
        print(f"{g}: {count}")

def main():
    contacts = load_contacts()
    
    while True:
        print("\n=== Contact Management System ===")
        print("1. Add contact")
        print("2. Search contacts")
        print("3. Update contact")
        print("4. Delete contact")
        print("5. Display all contacts")
        print("6. Export to CSV")
        print("7. Show statistics")
        print("8. Exit")
        
        choice = input("Enter your choice (1-8): ").strip()
        
        if choice == "1":
            contacts = add_contact(contacts)
            save_contacts(contacts)
        elif choice == "2":
            term = input("Enter name to search: ").strip()
            results = search_contacts(contacts, term)
            display_search_results(results)
        elif choice == "3":
            contacts = update_contact(contacts)
            save_contacts(contacts)
        elif choice == "4":
            contacts = delete_contact(contacts)
            save_contacts(contacts)
        elif choice == "5":
            display_all_contacts(contacts)
        elif choice == "6":
            export_to_csv(contacts)
        elif choice == "7":
            show_statistics(contacts)
        elif choice == "8":
            save_contacts(contacts)
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number from 1 to 8.")

if __name__ == "__main__":
    main()